In [38]:
import folium
from brails.utils import Importer
from brails.utils import PlotTools
import os
import pandas as pd

### Personal tests

In [2]:
API_KEY_DIR = r'C:\YOUR\OWN\PATH\TO\api_key.txt'
if os.path.exists(API_KEY_DIR): 
    with open(API_KEY_DIR,'r',encoding='utf-8') as file:
        api_key = file.readline().strip()

In [3]:
# Find by Area : 
LOCATION = 'Halifax, Canada'
LOCATION_TYPE = 'locationName' 

# Find by specific polygon : 
#LOCATION = (long1,Lat1,long2,lat2,long3,lat3,long4,lat4)
#LOCATION_TYPE = 'locationPolygon'



SCRAPER = "OSM_FootprintScraper"
INVENTORY_OUTPUT = 'Geo_data_OS2.geojson'

In [4]:
# Create an Importer instance:
importer = Importer()

# Create a region boundary:
region_boundary_class = importer.get_class('RegionBoundary')
region_boundary_object = region_boundary_class({'type': LOCATION_TYPE,
                                                'data': LOCATION})

In [ ]:
# Call a scraper to get geometric data for buildings:
scraper_class = importer.get_class(SCRAPER)
scraper = scraper_class({'length': 'ft'})
scraper_inventory = scraper.get_footprints(region_boundary_object)

# If a subset is required : 
#small_inventory = scrapper_inventory.get_random_sample(nsamples=20, seed=40)

In [ ]:
# Get GeoJSON data and save it:
geojson_data = scraper_inventory.write_to_geojson(INVENTORY_OUTPUT)

In [ ]:
# Extract building footprint coordinates from the inventory
inventory_footprints, _ = scraper_inventory.get_coordinates()

# Flatten the nested coordinate lists into a single list of points
all_coords = [coord for path in inventory_footprints for coord in path]

# Calculate the geographic center of all footprints for map centering
center_lat = sum(point[1] for point in all_coords) / len(all_coords)
center_lon = sum(point[0] for point in all_coords) / len(all_coords)

# Initialize an interactive map centered on the footprints
m = folium.Map(
    location=(center_lat, center_lon),
    tiles="cartodbpositron",  # Light, clean basemap style
    zoom_start=13
)

# Add building footprints as a GeoJSON layer with tooltips showing height and area
folium.GeoJson(
    geojson_data,
    name="geojson",
    tooltip=folium.GeoJsonTooltip(fields=['buildingheight', 'footprintArea'], sticky=False)
).add_to(m)

# Display the interactive map
m

### Aerial imagery

In [ ]:
google_satellite = importer.get_class('GoogleSatellite')()
images_satellite = google_satellite.get_images(
    scraper_inventory, 'tmp/satellite/')

#images_satellite.print_info()
#PlotTools.plot_images(images_satellite)

### Retrieve Street-level imagery

In [ ]:
google_street = importer.get_class('GoogleStreetview')({'apiKey': api_key})
images_street = google_street.get_images(scraper_inventory, 'tmp/street/')

#images_street.print_info()
#PlotTools.plot_images(images_street)

### Create building inventory

In [36]:
display(scraper_inventory.get_dataframe()[0])

,buildingheight,erabuilt,numstories,roofshape,footprintArea,chimneyExists,garageExists,Yearbuilt,Occupancy,ConstrType
index,,,,,,,,,,
0,NA,NA,NA,Gable,1083,0,0,1960,Residential,MAB
1,NA,NA,1,Gable,1581,0,0,2005,Residential,MAB
2,NA,NA,1,Gable,1058,0,0,1960,Residential,WOD
3,NA,NA,NA,Flat,1278,0,0,1960,Residential,WOD
4,NA,NA,2,Gable,952,0,0,1960,Residential,WOD
...,...,...,...,...,...,...,...,...,...,...
150,NA,NA,1,Flat,1962,0,0,1960,Other,MAS
151,NA,NA,2,Gable,1650,0,1,2005,Residential,WOD
152,NA,NA,NA,Flat,822,0,1,1960,Residential,WOD


In [11]:
# Local module import typo : a "dot" was missing. 
garage_detector_class = importer.get_class('GarageDetector')
garage_detector_classifier = garage_detector_class()
predictions = garage_detector_classifier.predict(images_street)


Checking the existence of garages for each building...
Default garage detector model in tmp/models/efficientdet-d4_garageDetector.pth loaded

Performing garage detections...


100%|████████████████████████████████████████████████████████████████████████████████| 155/155 [04:56<00:00,  1.91s/it]


Total execution time: 00:04:56.83


In [12]:
# Local module import typo : a "dot" was missing.
chimney_detector_class = importer.get_class('ChimneyDetector')
chimney_detector = chimney_detector_class()
predictions_chimney = chimney_detector.predict(images_street)


Checking the existence of chimneys for each building...
Default chimney detector model in tmp/models/efficientdet-d4_chimneyDetector.pth loaded

Performing chimney detections...


100%|████████████████████████████████████████████████████████████████████████████████| 155/155 [05:16<00:00,  2.04s/it]


Total execution time: 00:05:17.24


In [14]:
roof_shape_classifier_class = importer.get_class('RoofShapeClassifier')
roof_shape_classifier = roof_shape_classifier_class()
predictions_roof_shape = roof_shape_classifier.predict(images_satellite)


Default roof classifier model in tmp/models/roofTypeClassifier_v1.pth loaded


In [16]:
# Module "ConstructionTypeClassifier" required a small edit :
# Wrong format to querry the image classifier : images = data_dir , model_path = self.model_path, classes = self.classes
constype_classifier_class = importer.get_class('ConsTypeClassifier')
constype_classifier = constype_classifier_class()
predictions_const_type = constype_classifier.predict(images_street)

Default construction type classifier model at tmp/models/consTypeClassifier_v1.pth loaded
['MAB', 'MAS', 'RCC', 'STL', 'WOD'] from Const Type Class file
<brails.types.image_set.ImageSet object at 0x0000017034912710> tmp/models/consTypeClassifier_v1.pth


In [17]:
occupancy_classifier_class = importer.get_class('OccupancyClassifier')
occupancy_classifier = occupancy_classifier_class()
predictions_occupancies = occupancy_classifier.predict(images_street)


Default occupancy classifier model at tmp/models/OccupancyClassifier_v1.pth loaded


In [18]:
# Module "year_built_classifier" required a small edit : line 308 crashes because image path is modified.
consyear_classifier_class = importer.get_class('YearBuiltClassifier')
consyear_classifier = consyear_classifier_class()
predictions_constr_year = consyear_classifier.predict(images_street)

Default model checkpoint found at tmp\models.
Performing construction era classifications...


100%|████████████████████████████████████████████████████████████████████████████████████████| 155/155 [00:00<?, ?it/s]


In [26]:
# Add the recorded Features to the scraper inventory dataframe : 
for key, val in scraper_inventory.inventory.items():
    val.add_features({
        'chimneyExists': predictions_chimney[key],
        'garageExists' : predictions[key],
        'roofshape': predictions_roof_shape[key],
        'Yearbuilt' :predictions_constr_year[key],
        'Occupancy' :predictions_occupancies[key],
        'ConstrType':predictions_const_type[key], 
    })

In [ ]:
scraper_inventory.get_dataframe()

In [ ]:
# Fill the missing information :
knn_imputer_class = importer.get_class("KnnImputer")

imputer = knn_imputer_class(
    scraper_inventory,
    #exclude_features=['lat', 'fd_id']
)
imputed_inventory = imputer.impute()
imputed_inventory.get_dataframe()

In [29]:
inventory_df, geom_df, nbldg = imputed_inventory.get_dataframe()

In [32]:
inventory_df = inventory_df.drop(['buildingheight','erabuilt'], axis =1)
display(inventory_df)

,numstories,roofshape,footprintArea,chimneyExists,garageExists,Yearbuilt,Occupancy,ConstrType
index,,,,,,,,
0,2.0,Gable,1083,0,0,1960,Residential,MAB
1,1.0,Gable,1581,0,0,2005,Residential,MAB
2,1.0,Gable,1058,0,0,1960,Residential,WOD
3,1.0,Flat,1278,0,0,1960,Residential,WOD
4,2.0,Gable,952,0,0,1960,Residential,WOD
...,...,...,...,...,...,...,...,...
150,1.0,Flat,1962,0,0,1960,Other,MAS
151,2.0,Gable,1650,0,1,2005,Residential,WOD
152,2.0,Flat,822,0,1,1960,Residential,WOD


In [ ]:
display(geom_df)

In [35]:
print(nbldg)

155


In [ ]:
summary_df = pd.concat([inventory_df,geom_df],axis = 1)
display(summary_df)

In [41]:
summary_df.to_excel('BRAILS_inventory.xlsx')